# **Load and Prepare HR Employee Attrition Data**

## Objectives

This notebook has been updated to reflect the **HR Employee Attrition** project data source and the required preparation steps were followed.

It main goals are to:

- load the **HR Employee Attrition dataset**
- document the **data collection** approach

- perform **basic data cleaning and validation**
- **encode categorical variables**
- **normalise key numerical features**
- save cleaned outputs for later analysis and dashboard development

## Important note on the data source

The project documentation states that the analysis uses an **HR Employee Attrition dataset** containing employee demographics, employment details, compensation data, satisfaction measures, and the target field **Attrition (Yes/No)**.


`dataset/raw/hr_employee_attrition.csv`



---

# Change working directory

   
To make relative file paths work correctly, we move from the notebook folder to the project root.


In [ ]:
import os

current_dir = os.getcwd()
current_dir


In [ ]:
os.chdir(os.path.dirname(current_dir))
print("Working directory updated to project root")


In [ ]:
current_dir = os.getcwd()
current_dir


---
# Initial Setup and Data Collection

## Data Collection

- The dataset used is the **HR Employee Attrition dataset**, selected specifically to address the business problem of understanding and reducing employee turnover.
- The dataset includes employee-level data such as demographics, employment details, compensation, satisfaction metrics, and attrition status.
- It has been sourced as an **open-source dataset** suitable for analytical and educational use.https://www.kaggle.com/code/encodedanand/employee-attrition/input
- The dataset reflects a **real-world HR analytics scenario**, where organisations aim to reduce attrition and improve workforce retention.
- No personally identifiable information (PII) is included, ensuring compliance with **ethical and GDPR considerations**.
- A known limitation is that the dataset represents a **static snapshot**, meaning trends over time cannot be analysed.

## Import libraries


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# Load the HR attrition dataset
# Update the filename if your CSV uses a slightly different name.

file_path = "dataset/raw/hr_employee_attrition.csv"
df = pd.read_csv(file_path)
df.head()


## Quick dataset inspection

The first inspection step checks that the file loaded correctly and gives an overview of the columns, data types, and completeness of the dataset.


In [ ]:
df.info()


---

# Dataset Content

Based on the project brief, the dataset is expected to include fields such as:

- **Demographics**: `Age`, `Gender`
- **Employment details**: `Department`, `JobRole`, `YearsAtCompany`
- **Compensation**: `MonthlyIncome`
- **Satisfaction metrics**: `JobSatisfaction`, `WorkLifeBalance`
- **Target variable**: `Attrition`

The exact column names can vary depending on the source file, so the next step is to inspect all column names directly from the dataset.


In [ ]:
df.columns.tolist()


# Data Cleaning & Preparation

The following steps were applied to ensure data quality and consistency:

### 1. Handling missing values
- Identified missing or inconsistent values
- Applied appropriate imputation strategies (median for numerical, 'Unknown' for categorical)

### 2. Removing duplicates
- Removed redundant records to prevent bias in analysis

### 3. Encoding categorical variables
- Converted categorical variables into numerical format using one-hot encoding

### 4. Normalising key features
- Applied MinMax scaling to ensure consistent ranges across numerical features

These steps ensure the dataset is reliable, consistent, and ready for analysis and modelling.


## 1. Check missing values and duplicates


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
df.duplicated().sum()


## 2. Handle missing values

A mixed strategy is used:

- **categorical columns**: fill missing values with `"Unknown"`
- **numerical columns**: fill missing values with the median

This keeps the dataset usable while limiting distortion from extreme values.


In [ ]:
df_clean = df.copy()

categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

for col in numerical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean.isna().sum().sort_values(ascending=False).head(20)


## 3. Remove duplicate rows

Duplicate rows can distort summary statistics and model outputs, so they are removed where present.


In [ ]:
df_clean = df_clean.drop_duplicates()
df_clean.shape


## 4. Encode categorical variables

Categorical variables are transformed into numeric format so they can be used in downstream analysis and modelling.

The notebook uses **one-hot encoding** with `drop_first=True` to reduce redundancy.


In [ ]:
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)
df_encoded.head()


## 5. Normalise key numerical features

Normalisation is applied to numerical columns so that variables measured on different scales can be compared more easily.

This is especially useful for later modelling or distance-based methods.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_prepared = df_encoded.copy()

scaler = MinMaxScaler()
available_numerical_cols = [col for col in numerical_cols if col in df_prepared.columns]

df_prepared[available_numerical_cols] = scaler.fit_transform(df_prepared[available_numerical_cols])

df_prepared[available_numerical_cols].head()


## 6. Validation checks

These checks confirm that the prepared dataset is suitable for the next stage of analysis.


In [ ]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Prepared shape:", df_prepared.shape)
print("\nMissing values remaining in prepared dataset:", int(df_prepared.isna().sum().sum()))


---

# Saving Cleaned Data

Two outputs are saved:

1. **Cleaned dataset** before encoding  
2. **Prepared dataset** after encoding and normalisation  

This supports traceability and good data management practice.


In [ ]:
os.makedirs("dataset/processed", exist_ok=True)

df_clean.to_csv("dataset/processed/hr_employee_attrition_cleaned.csv", index=False)
df_prepared.to_csv("dataset/processed/hr_employee_attrition_prepared.csv", index=False)

print("Saved:")
print("- dataset/processed/hr_employee_attrition_cleaned.csv")
print("- dataset/processed/hr_employee_attrition_prepared.csv")


---

# Conclusion

This notebook has been updated to reflect the **HR Employee Attrition** data source and the required preparation steps.

Completed in this notebook:

- documented the **data collection** rationale
- aligned the notebook to the **HR attrition dataset**
- checked data structure, missing values, and duplicates
- handled missing values
- encoded categorical variables
- normalised key numerical features
- saved processed datasets for the next analysis stage

This now provides a cleaner foundation for EDA, hypothesis testing, and dashboard development.
